# N039 · 快速排序与划分不变量

**目标：** 保持划分区域含义并处理重复元素。

**先修：** N037, N038。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Lomuto/Hoare区别；三路划分；随机枢轴；最坏复杂度；递归深度。

## 从问题到算法

重复值会让朴素二路快排产生不必要递归。三路划分维护“小于pivot、等于pivot、尚未处理、大于pivot”四段。遇到大值与未处理区末尾交换后，当前位置的新元素仍未分类，不能递增扫描指针。本章显式栈避免Python递归深度问题。

## 最小实现

**本章接口：** `partition3(a, lo, hi)`、`quicksort(a, rng)`

In [1]:
def partition3(a,lo,hi):
    """Partition half-open [lo, hi); return equal block [lt, gt)."""
    if not 0<=lo<hi<=len(a): raise ValueError('requires nonempty valid range')
    pivot=a[lo]; lt=i=lo; gt=hi
    while i<gt:
        if a[i]<pivot:
            a[lt],a[i]=a[i],a[lt]; lt+=1; i+=1
        elif a[i]>pivot:
            gt-=1; a[gt],a[i]=a[i],a[gt]
        else: i+=1
    return lt,gt

def quicksort(a,rng):
    stack=[(0,len(a))]
    while stack:
        lo,hi=stack.pop()
        if hi-lo<2: continue
        p=rng.randrange(lo,hi); a[lo],a[p]=a[p],a[lo]
        lt,gt=partition3(a,lo,hi)
        stack.append((lo,lt)); stack.append((gt,hi))
    return a

## 正确性、前提与复杂度

每次循环使未处理区长度减少1，并维持已分类区关系；划分后只需排序小于和大于两段。随机pivot不改变正确性，只影响复杂度分布。

**代价：** 随机独立pivot下期望 O(n log n)，最坏 O(n²)；显式栈最坏 O(n)，原地划分 O(1)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,1,3,5,2,3,4]; original=a[:]; lt,gt=partition3(a,0,len(a))
show_table(['原数组','<pivot','=pivot','>pivot'],[(original,a[:lt],a[lt:gt],a[gt:])])

原数组,<pivot,=pivot,>pivot
"[3, 1, 3, 5, 2, 3, 4]","[1, 2]","[3, 3, 3]","[4, 5]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
from collections import Counter
rng=Random(39)
for n in range(80):
    a=[rng.randrange(-3,4) for _ in range(n)]
    assert quicksort(a[:],Random(7))==sorted(a)
    if a:
        b=a[:]; pivot=b[0]; lt,gt=partition3(b,0,n)
        assert all(x<pivot for x in b[:lt]) and all(x==pivot for x in b[lt:gt]) and all(x>pivot for x in b[gt:])
        assert Counter(a)==Counter(b)
assert quicksort([2]*1000,Random(1))==[2]*1000
print('N039: 所有本章断言通过')

N039: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造固定枢轴退化输入。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 改为先递归较短区间。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 912. Sort an Array（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。